In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import MinMaxScaler, FunctionTransformer
import optuna
from sklearn.svm import SVC
from sklearn.metrics import recall_score, f1_score, roc_auc_score


DATA_PATH = os.path.join('..', 'vr_legibility_train.csv')

c:\Users\psgve\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. 데이터 로드 및 전처리
df = pd.read_csv(DATA_PATH)[["x_theta", "y_theta", "TextSize", "Label"]]
df = df[df['TextSize'] == 17.3]
df = df.reset_index(drop=True)
print(df)

# 특성(X)과 타겟(y) 분리
X = df.drop(['Label', 'TextSize'] , axis=1)
y = df['Label']

# 학습/테스트 데이터 분할 9:1 (Stratified Sampling)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

# 학습/테스트 데이터 분할 (StratifiedKFold 적용)
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

     x_theta  y_theta  TextSize  Label
0     -48.47    22.07      17.3      1
1      30.55   -27.38      17.3      1
2     -15.17     5.07      17.3      0
3      43.25   -31.73      17.3      1
4     -34.55    37.93      17.3      1
..       ...      ...       ...    ...
602   -16.59   -51.73      17.3      1
603   -30.41   -46.76      17.3      1
604    17.66   -33.95      17.3      0
605   -41.72    41.96      17.3      1
606    46.62    19.24      17.3      1

[607 rows x 4 columns]


In [3]:
# 2. 파이프라인 (스케일링 + 모델) 구축
# ---------------------------------------------------------
preprocessor = make_column_transformer(
    (MinMaxScaler(), ['x_theta', 'y_theta']),
    #(make_pipeline(FunctionTransformer(np.log1p), MinMaxScaler()), ['FontSize'])
)

# 최종 파이프라인 (기존 Optuna의 'svm__C' 파라미터 이름을 그대로 쓰기 위해 Pipeline 유지)
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('svm', SVC(class_weight='balanced', random_state=42))
])

In [4]:
# 3. 교차 검증 객체 생성 및 Optuna를 활용한 하이퍼파라미터 탐색 레인지 설정
def objective(trial, X_train, y_train, pipeline):
    kernel = trial.suggest_categorical('svm__kernel', ['rbf', 'linear', 'poly'])
   
    params = {'svm__kernel': kernel}
        
    if kernel == 'rbf':
        params['svm__C'] = trial.suggest_float('svm__C', 0.1, 1000.0, log=True)
        params['svm__gamma'] = trial.suggest_float('svm__gamma', 1e-3, 10.0, log=True)
            
    elif kernel == 'linear':
        params['svm__C'] = trial.suggest_float('svm__C', 1e-3, 10.0, log=True)
            
    elif kernel == 'poly':
        params['svm__C'] = trial.suggest_float('svm__C', 0.1, 1.0, log=True)
        params['svm__gamma'] = trial.suggest_float('svm__gamma', 1e-3, 1.0, log=True)
        params['svm__degree'] = trial.suggest_int('svm__degree', 2, 3)
            
    # 파이프라인에 파라미터 적용
    pipeline.set_params(**params)
        
    scores = cross_val_score(
    pipeline, X_train, y_train, 
    cv=inner_cv, scoring='f1', n_jobs=-1,
    verbose=0
    )
    return scores.mean()

In [5]:
# 4. Outer CV 루프에서 Optuna 최적화 실행
import warnings
warnings.filterwarnings('ignore') # 경고 메시지 무시
optuna.logging.set_verbosity(optuna.logging.WARNING) # 로그가 너무 많이 뜨는 것을 방지 

f1_scores = []
rec_scores = []
auc_scores = []

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_train, y_train)):
    print(f"\n--- Outer Fold {fold_idx + 1} / 10 ---")
    X_train_val, X_test_val = X_train.iloc[train_idx], X_train.iloc[test_idx]
    y_train_val, y_test_val = y_train.iloc[train_idx], y_train.iloc[test_idx]

    # Optuna Study 생성 및 최적화 실행
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(lambda trial: objective(trial, X_train_val, y_train_val, pipeline), n_trials=120, n_jobs=1) 

    best_params_inner = study.best_params
    best_score_inner = study.best_value
    
    print("-" * 40)
    print(f"Best Hyper Params: {best_params_inner}")
    print(f"Best Inner CV F1:  {best_score_inner:.4f}")

    # 하이퍼파라미터로 모델 학습
    best_model_inner = pipeline.set_params(**best_params_inner)
    best_model_inner.fit(X_train_val, y_train_val)

    # 검증 데이터 평가
    y_pred = best_model_inner.predict(X_test_val)
    y_score = best_model_inner.decision_function(X_test_val)

    f1 = f1_score(y_test_val, y_pred)
    acc = recall_score(y_test_val, y_pred)
    roc_auc = roc_auc_score(y_test_val, y_score)

    #최고성능 하이퍼파라미터로 갱신
    if fold_idx == 0:
        best_param=best_params_inner
        best_f1=f1
    elif f1 > best_f1:
        best_param=best_params_inner
        best_f1=f1

    f1_scores.append(f1)
    rec_scores.append(acc)
    auc_scores.append(roc_auc)
    print(f"\n★ Outer fold Score:: F1 Score: {f1:.4f}, Recall: {acc:.4f}, ROC AUC: {roc_auc:.4f}")

print("\n"+"-" * 10 + " FINAL RESULTS " + "-" * 10)
print(f"최종 10-Fold 평균 F1 Score: {np.mean(f1_scores):.4f} (± {np.std(f1_scores):.4f})")
print(f"최종 10-Fold 평균 Recall: {np.mean(rec_scores):.4f} (± {np.std(rec_scores):.4f})")
print(f"최종 10-Fold 평균 ROC AUC: {np.mean(auc_scores):.4f} (± {np.std(auc_scores):.4f})")
print(f"최고 F1 하이퍼파라미터: {best_param}")


--- Outer Fold 1 / 10 ---
----------------------------------------
Best Hyper Params: {'svm__kernel': 'rbf', 'svm__C': 12.636204746088795, 'svm__gamma': 1.8622437185355554}
Best Inner CV F1:  0.9279

★ Outer fold Score:: F1 Score: 0.9296, Recall: 0.9706, ROC AUC: 0.9482

--- Outer Fold 2 / 10 ---
----------------------------------------
Best Hyper Params: {'svm__kernel': 'rbf', 'svm__C': 1.0585171385389132, 'svm__gamma': 4.4301397993185825}
Best Inner CV F1:  0.9305

★ Outer fold Score:: F1 Score: 0.8732, Recall: 0.9118, ROC AUC: 0.9216

--- Outer Fold 3 / 10 ---
----------------------------------------
Best Hyper Params: {'svm__kernel': 'rbf', 'svm__C': 18.612873211585693, 'svm__gamma': 1.6214929900885873}
Best Inner CV F1:  0.9174

★ Outer fold Score:: F1 Score: 0.9714, Recall: 0.9714, ROC AUC: 0.9929

--- Outer Fold 4 / 10 ---
----------------------------------------
Best Hyper Params: {'svm__kernel': 'rbf', 'svm__C': 357.8867385682977, 'svm__gamma': 0.9758422140711235}
Best Inner 

In [6]:
# 5. 최종 모델 학습 및 저장
import joblib

print(" [최종 모델 학습 시작]")
print(X.columns)
print("=========================================")

#확률 옵션 키기
best_param['svm__probability'] = True

final_model = pipeline.set_params(**best_param)
final_model.fit(X_train, y_train) 

# 검증 데이터 평가
final_pred = final_model.predict(X_test)
final_roc_auc = final_model.decision_function(X_test)

final_f1 = f1_score(y_test, final_pred)
final_rec = recall_score(y_test, final_pred)
roc_auc = roc_auc_score(y_test, final_roc_auc)
print(f"최종 모델 F1 Score on Test Set: {final_f1:.4f}")
print(f"최종 모델 Recall on Test Set: {final_rec:.4f}")
print(f"최종 모델 AUC Score on Test Set: {roc_auc:.4f}")

# 학습이 완료된 final_model 객체를 하드디스크에 파일로 저장합니다.
joblib.dump(final_model, os.path.join('..', 'vr_legibility_svm_model17.pkl'))
print("✅ 최종 모델이 'vr_legibility_svm_model17.pkl' 파일로 저장되었습니다!")

 [최종 모델 학습 시작]
Index(['x_theta', 'y_theta'], dtype='object')
최종 모델 F1 Score on Test Set: 0.9067
최종 모델 Recall on Test Set: 0.8947
최종 모델 AUC Score on Test Set: 0.9268
✅ 최종 모델이 'vr_legibility_svm_model17.pkl' 파일로 저장되었습니다!


In [7]:
# 6. 모델 배포 후 예측 (새로운 데이터에 대한 예측)
import joblib
import pandas as pd
import os
from sklearn.metrics import recall_score, f1_score, roc_auc_score
# 1. 저장해둔 모델 불러오기 
deployed_model = joblib.load(os.path.join('..', 'vr_legibility_svm_model17.pkl'))

# 2. 새 데이터 불러오기
"""
new_data = pd.DataFrame({
    'TextSize': [180.0, 160.0],
    'x_theta': [1250.5, -450.2],
    'y_theta': [45.2, 210.8],
    "Label" :[1, 1]
})
"""
new_data = pd.read_csv(os.path.join('..', 'coordinates.csv'))[["x_theta", "y_theta"]]

X_new = new_data

# 3. 새로운 데이터 예측
predictions = deployed_model.predict(X_new)
probabilities = deployed_model.predict_proba(X_new)[:, 1]
score = deployed_model.decision_function(X_new)

print("새로운 데이터 예측 완료! (총 {}개)".format(len(predictions)))
new_data['Prediction'] = predictions          # 0과 1 예측값 열 추가
new_data['Probability'] = probabilities       # 1일 확률 열 추가
print("\n--- 예측 결과 확인 ---")
print(new_data.head(10)) # 결과가 잘 들어갔는지 상위 10개 확인
new_data.to_csv('svm_new_data_predictions17.csv', index=False) # 예측 결과를 CSV 파일로 저장
"""
rec = recall_score(y_true, predictions)
f1 = f1_score(y_true, predictions)
auc = roc_auc_score(y_true, score)
print("\n" + "-" * 40)
print(f"새로운 데이터에 대한 Recall: {rec:.4f}")
print(f"새로운 데이터에 대한 F1 Score: {f1:.4f}")
print(f"새로운 데이터에 대한 ROC-AUC: {auc:.4f}")
"""

새로운 데이터 예측 완료! (총 10767개)

--- 예측 결과 확인 ---
   x_theta  y_theta  Prediction  Probability
0      -55      -53           1     0.997120
1      -55      -52           1     0.996778
2      -55      -51           1     0.996406
3      -55      -50           1     0.996002
4      -55      -49           1     0.995565
5      -55      -48           1     0.995095
6      -55      -47           1     0.994592
7      -55      -46           1     0.994054
8      -55      -45           1     0.993484
9      -55      -44           1     0.992881


'\nrec = recall_score(y_true, predictions)\nf1 = f1_score(y_true, predictions)\nauc = roc_auc_score(y_true, score)\nprint("\n" + "-" * 40)\nprint(f"새로운 데이터에 대한 Recall: {rec:.4f}")\nprint(f"새로운 데이터에 대한 F1 Score: {f1:.4f}")\nprint(f"새로운 데이터에 대한 ROC-AUC: {auc:.4f}")\n'